<h3 style="color: red;">CONECT DB</h3>

In [ ]:
import mysql.connector # pip install mysql-connector-python

conn = mysql.connector.connect(
    host="localhost",       
    user="root",     
    password="Sql654**",
    database="mysql_omdb"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS mysql_omdb")
cursor.execute("USE mysql_omdb")

if conn.is_connected():
    print("Success!")
    print(conn.get_server_info()) # infos do server
else:
    print("Error!")

Success!
8.0.45


C:\Users\isabella.rocha\AppData\Local\Temp\ipykernel_17428\537116498.py:17: DeprecationWarning: Call to deprecated function get_server_info. Reason: 
    The property counterpart 'server_info' should be used instead.

  print(conn.get_server_info()) # infos do server


<h3 style="color: red;">TABLE MOVIES DB</h3>

In [15]:
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS movies (
    id INT AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(255) UNIQUE,
    year VARCHAR(10),
    runtime VARCHAR(50),
    genre VARCHAR(100),
    director VARCHAR(100)
)
""")

conn.commit() #connection in sqlite

<h3 style="color: red;">TABLE USERS DB</h3>

In [14]:
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS users (
  id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(100) UNIQUE,
    password VARCHAR(255)
);
""")
               
conn.commit()

<h3 style="color: red;">TABLE RATINGS DB</h3>

In [13]:
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS ratings (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    movie_id INT,
    rating FLOAT,
    FOREIGN KEY (user_id) REFERENCES users(id),
    FOREIGN KEY (movie_id) REFERENCES movies(id),
    UNIQUE(user_id, movie_id)
);
""")

conn.commit()

<h3 style="color: red;">TABLE COMMENTS DB</h3>

In [11]:
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS comments (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    movie_id INT,
    comment VARCHAR(500),
    FOREIGN KEY (user_id) REFERENCES users(id),
    FOREIGN KEY (movie_id) REFERENCES movies(id)
    );
""")

conn.commit()

<h3 style="color: red;">ADD MOVIE DB</h3>

In [6]:
import requests

apikey = "9a51a7ae"
movie = "Ghostbusters 2"    # CHANGE THE MOVIE HERE!!!

url = f"https://www.omdbapi.com/?apikey={apikey}&t={movie}"

response = requests.get(url)
data = response.json()

if data["Response"] == "True":

    title = data["Title"]
    year = data["Year"]
    runtime = data["Runtime"]
    genre = data["Genre"]
    director = data["Director"]
 
    cursor.execute("SELECT * FROM movies WHERE LOWER(title) = LOWER(%s)", (title,))
    exists = cursor.fetchone()

    if not exists:

        cursor.execute("""
            INSERT INTO movies
            (title, year, runtime, genre, director)
            VALUES (%s, %s, %s, %s, %s)
        """, (title, year, runtime, genre, director))

        conn.commit()
        print("Movie added!")

    else:
        print("Movie already exists.")

else:
    print("Error:", data["Error"])



Movie already exists.


<h3 style="color: red;">SHOW DB/TABLES/RESULTS</h3>

In [7]:
cursor.execute("USE mysql_omdb")

cursor.execute("SHOW DATABASES")
print(cursor.fetchall())

cursor.execute("SHOW TABLES")
print(cursor.fetchall())

cursor.execute("SELECT * FROM movies")
result = cursor.fetchall()
for row in result:
    print(row)

[('information_schema',), ('mysql',), ('mysql_omdb',), ('performance_schema',), ('sakila',), ('sys',), ('world',)]
[('comments',), ('movies',), ('ratings',), ('users',)]
(1, 'Ghostbusters', '1984', '105 min', 'Action, Comedy, Fantasy', 'Ivan Reitman', 'Unrated')
(2, 'Time Is But a Window: Ghostbusters 2 and Beyond', '2014', '17 min', 'Short', 'N/A', 'Unrated')


In [ ]:
import tkinter as tk
from tkinter import simpledialog, messagebox
import requests

def register():
    username = simpledialog.askstring("Register", "Username:")
    password = simpledialog.askstring("Register", "Password:")
    
    cursor.execute("SELECT * FROM users WHERE username=%s", (username,))
    if cursor.fetchone():
        messagebox.showerror("Erro", "Usuário já existe")
        return
    
    cursor.execute("INSERT INTO users (username, password) VALUES (%s,%s)", (username, password))
    conn.commit()
    messagebox.showinfo("Sucesso", "Usuário registrado!")

def login():
    global current_user
    username = simpledialog.askstring("Login", "Username:")
    password = simpledialog.askstring("Login", "Password:")
    
    cursor.execute("SELECT * FROM users WHERE username=%s AND password=%s", (username, password))
    user = cursor.fetchone()
    if user:
        current_user = user[0]
        messagebox.showinfo("Sucesso", f"Login feito! ID: {current_user}")
    else:
        messagebox.showerror("Erro", "Login inválido")

def list_movies():
    cursor.execute("SELECT * FROM movies")
    movies = cursor.fetchall()
    text = "\n".join([f"{m[0]} - {m[1]} ({m[2]})" for m in movies])
    messagebox.showinfo("Filmes", text or "Nenhum filme cadastrado")

def add_movie():
    title = simpledialog.askstring("Adicionar Filme", "Nome do filme:")
    url = f"https://www.omdbapi.com/?apikey=9a51a7ae&t={title}"
    data = requests.get(url).json()
    
    if data["Response"] == "True":
        cursor.execute("SELECT * FROM movies WHERE title=%s", (data["Title"],))
        if cursor.fetchone():
            messagebox.showerror("Erro", "Filme já existe")
            return
        
        cursor.execute(
            "INSERT INTO movies (title, year, runtime, genre, director) VALUES (%s,%s,%s,%s,%s)",
            (data["Title"], data["Year"], data["Runtime"], data["Genre"], data["Director"])
        )
        conn.commit()
        messagebox.showinfo("Sucesso", "Filme adicionado!")
    else:
        messagebox.showerror("Erro", "Filme não encontrado")

# ---------- Interface ----------
current_user = None
root = tk.Tk()
root.title("Movie System")

tk.Button(root, text="Register", command=register).pack(pady=5)
tk.Button(root, text="Login", command=login).pack(pady=5)
tk.Button(root, text="Ver Filmes", command=list_movies).pack(pady=5)
tk.Button(root, text="Adicionar Filme", command=add_movie).pack(pady=5)

root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\isabella.rocha\AppData\Local\Programs\Python\Python314\Lib\site-packages\mysql\connector\connection_cext.py", line 772, in cmd_query
    self._cmysql.query(
    ~~~~~~~~~~~~~~~~~~^
        query,
        ^^^^^^
    ...<3 lines>...
        query_attrs=self.query_attrs,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
_mysql_connector.MySQLInterfaceError: Unknown column 'username' in 'where clause'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\isabella.rocha\AppData\Local\Programs\Python\Python314\Lib\tkinter\__init__.py", line 2093, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\isabella.rocha\AppData\Local\Temp\ipykernel_17428\3527248574.py", line 20, in register
    cursor.execute("SELECT * FROM users WHERE username=%s", (username,))
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^